# Lentils × Dinomaly: training tutorial (CIR front-end)

A lentil conveyor should carry nothing but lentils. In practice it also carries stones,
aluminium shards, paper snippets, rubber, and the occasional fly. Cataloguing every possible
foreign object is hopeless, which is what makes this an **anomaly detection** problem: learn
what normal lentils look like, and flag whatever deviates.

Dinomaly does exactly that. It is reconstruction-based and fully unsupervised: a frozen DINOv2
vision transformer extracts features, and a small decoder learns to reconstruct those features
**on normal frames only**. At inference, regions the decoder cannot reconstruct are the
anomalies. No foreign object is ever seen during training.

This notebook trains Dinomaly on the 61-band VNIR lentils dataset with a **CIR (color-infrared)
front-end**: a CIR selector maps NIR 860 / Red 670 / Green 560 nm onto the 3 R/G/B channels,
so the ViT runs in its native image domain while the near-infrared band, where lentils and
many foreign objects separate cleanly, drives the red channel.

**Pipeline at a glance:**

```
AnomalyDataNode ──► MinMaxNormalizer ──► CIRSelector(NIR 860 / Red 670 / Green 560 nm)
     ──► DinomalyDetector ──► {QuantileBinaryDecider, AnomalyDetectionMetrics,
         AnomalyAUROCMetrics} ──► TensorBoardMonitorNode
```

Sibling notebooks: `lentils_rgb_train_tutorial.ipynb` (RGB front-end),
`lentils_adaclip_bands_train_tutorial.ipynb` (AdaCLIP-selected bands), and
`lentils_inference_tutorial.ipynb` (evaluation with per-class AUROC).

> **Prerequisites**
>
> 1. Install cuvis-ai-dinomaly with the examples extra: `uv sync --extra examples`.
> 2. From the repo root, launch the notebook with `uv run jupyter lab`.

In [ ]:
# Colab bootstrap: no-op when running locally
try:
    import google.colab  # noqa: F401

    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    print("In Colab Env")
    %pip install -q cuvis-ai cuvis-ai-dinomaly "cuvis-ai-dataloader[cu3s,coco]"

    import torch

    if not torch.cuda.is_available():
        print(
            "WARNING: No GPU detected. Switch via Runtime > Change runtime type > T4 GPU. "
            "Dinomaly training on CPU is very slow."
        )

In [ ]:
from pathlib import Path

import torch
from cuvis_ai.node.anomaly_visualization import MaskOverlayNode
from cuvis_ai.node.channel_selector import CIRSelector
from cuvis_ai.node.data import AnomalyDataNode, CU3SDataNode
from cuvis_ai.node.deciders.binary_decider import QuantileBinaryDecider
from cuvis_ai.node.metrics import AnomalyDetectionMetrics
from cuvis_ai.node.monitor import TensorBoardMonitorNode
from cuvis_ai.node.normalization import MinMaxNormalizer
from cuvis_ai.node.video import ToVideoNode
from cuvis_ai_core.data.public_datasets import PublicDatasets
from cuvis_ai_core.pipeline.pipeline import CuvisPipeline
from cuvis_ai_core.training import Predictor
from cuvis_ai_core.utils import restore_trainrun
from cuvis_ai_dataloader.data import MultiNpzDataModule
from cuvis_ai_dataloader.data.npz_converter import convert_universe
from cuvis_ai_schemas.pipeline import PipelineMetadata
from cuvis_ai_schemas.training import (
    CallbacksConfig,
    DataConfig,
    DataSplitConfig,
    ModelCheckpointConfig,
    OptimizerConfig,
    TrainingConfig,
    TrainRunConfig,
)
from IPython.display import Video, display
from loguru import logger

from cuvis_ai_dinomaly.node.auroc_metrics import AnomalyAUROCMetrics
from cuvis_ai_dinomaly.node.dinomaly_detector import DinomalyDetector
from cuvis_ai_dinomaly.node.dinomaly_train_loss_bridge import DinomalyTrainLossBridge

In [ ]:
import sys

# Keep the notebook output readable: log at INFO and above. Nodes emit per-step progress
# (e.g. the TensorBoard monitor) at DEBUG, a firehose across hundreds of val/test steps;
# raise the floor to INFO here.
logger.remove()
logger.add(sys.stderr, level="INFO")

device = torch.device("cpu")
# Pick the best available torch device
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
logger.info(f"Using device {device}")

## 1 · Fetch + prepare the dataset

The XMR Industrial Foreign Object Detection (Lentils) dataset (~57 GB) lives on Hugging Face
Hub: 15 merged cu3s sessions recorded on three acquisition days, 61 spectral bands (430 to
910 nm) per pixel, and pixel-level COCO masks for 7 foreign-object classes.

Dinomaly trains on normals only, so the dataset ships a dedicated split manifest
(`universe.csv` + `splits/dinomaly.json`):

| split | frames | anomalous | role |
|---|---|---|---|
| train | 308 | 0 | Dinomaly training |
| val | 148 | 84 | calibration / threshold |
| test | 180 | 112 | evaluation |
| adaclip_train | 500 | 500 | held out for the supervised baseline |

Two steps, each skipped when its output already exists:

1. **Fetch** the raw dataset with `PublicDatasets.download_dataset` (skips when the folder is
   already on disk).
2. **Convert** the manifest's frames to per-frame NPZ with `convert_universe` from
   cuvis-ai-dataloader. It writes a `universe.csv` (the sample universe, listing which `.npz`
   holds which `(source, frame)`) and a `splits.json` (core's `DataSplitConfig`, the same
   selector-based split file the CuvisNEXT split designer loads and saves, so one file drives
   training here and in the GUI). Re-running is cheap: already-converted frames are reused.

In [ ]:
LENTILS_DATASET_NAME = "industrial_fod_lentils"
NPZ_DIR = Path("outputs/npz_local")
SPLITS_JSON = NPZ_DIR / "splits.json"
UNIVERSE_CSV = NPZ_DIR / "universe.csv"

dataset_dir = Path("/content/data") if IN_COLAB else Path("../../data")
raw_dir = dataset_dir / "XMR_Industrial_Foreign_Object_Detection_Lentils"

if SPLITS_JSON.is_file() and UNIVERSE_CSV.is_file():
    print(f"Split artifacts already present: {SPLITS_JSON}")
else:
    PublicDatasets.download_dataset(
        LENTILS_DATASET_NAME, download_path=str(dataset_dir), force=False
    )

In [ ]:
if not (SPLITS_JSON.is_file() and UNIVERSE_CSV.is_file()):
    result = convert_universe(
        raw_dir / "universe.csv",
        raw_dir,
        NPZ_DIR,
        splits_json=raw_dir / "splits" / "dinomaly.json",
        out_universe_csv=UNIVERSE_CSV,
        out_splits_json=SPLITS_JSON,
    )
    SPLITS_JSON, UNIVERSE_CSV = result.splits_json, result.universe_csv

print("Splits JSON: ", SPLITS_JSON)
print("Universe CSV:", UNIVERSE_CSV)

## 2 · How Dinomaly works

Dinomaly is reconstruction-based anomaly detection on ViT features:

- A **frozen DINOv2 ViT-B/14 encoder** turns each frame into patch-token features. It is
  never trained here; its weights download automatically on first use.
- A small **bottleneck MLP** and an 8-block **transformer decoder** are the trainable parts.
  They learn to reconstruct the encoder's features on normal lentils.
- The **anomaly map** is the cosine distance between encoder and decoder features: regions the
  decoder cannot reconstruct (it never saw them in training) score high. Out come a pixel map
  (`scores`, one value per pixel) and a per-frame `anomaly_score`.

Encoder, bottleneck, and decoder live inside the single `DinomalyDetector` node, following the
wrapper-first convention for pretrained networks: they are hyperparameters of one node, not
separate nodes. What you mix and match at pipeline level sits around it: the selector
front-end (this notebook uses a CIR NIR/Red/Green mapping; the siblings use fixed RGB
wavelengths and AdaCLIP-selected bands), the decider, and the metric nodes.

## 3 · Tutorial configuration

Edit these variables to customise the run.

- **`MAX_EPOCHS`**: 1 gives a quick smoke run that exercises the whole path end to end. Set
  20 to 50 for a real model.
- **`IMAGE_SIZE`**: square side the detector works at; a multiple of the ViT patch size 14.
- **`output_dir`**: everything this notebook produces lands here (pipeline yaml, trainrun
  yaml, checkpoints, TensorBoard logs, preview video, trained weights).

In [ ]:
MAX_EPOCHS = 1  # 20-50 for a real run
IMAGE_SIZE = 448  # multiple of 14

output_dir = Path("outputs/lentils_cir_run")
output_dir.mkdir(parents=True, exist_ok=True)

pipeline_yaml_path = output_dir / "dinomaly_lentils_cir.yaml"
trainrun_yaml_path = output_dir / "trainrun.yaml"
preview_video_path = output_dir / "test_preview.mp4"

print(f"Epochs:       {MAX_EPOCHS}")
print(f"Image size:   {IMAGE_SIZE}")
print(f"Splits JSON:  {SPLITS_JSON}")
print(f"Universe CSV: {UNIVERSE_CSV}")
print(f"Output dir:   {output_dir}")

## 4 · Preview the data, as a pipeline

Before training, look at what the model will see. Even the preview is a Cuvis.AI pipeline:
a `CU3SDataNode`, the same `CIRSelector` the training pipeline uses, a
`MaskOverlayNode` that tints the ground-truth foreign objects, and a `ToVideoNode` sink.
Computation happens in nodes; the notebook only plays the result.

```
CU3SDataNode ──► CIRSelector ──► MaskOverlayNode ──► ToVideoNode
```

Two cells: build the pipeline and show its graph, then render the whole test split to an
annotated video, foreign objects tinted red and normal frames passing through untouched, and
play it inline.

In [ ]:
preview_pipeline = CuvisPipeline("lentils_cir_preview")

preview_data = CU3SDataNode(name="data")
preview_selector = CIRSelector(
    nir_nm=860.0,
    red_nm=670.0,
    green_nm=560.0,
    norm_mode="running",
    running_warmup_frames=0,
    freeze_running_bounds_after_frames=20,
    name="cir_selector",
)
preview_overlay = MaskOverlayNode(alpha=0.4, overlay_color=(1.0, 0.0, 0.0), name="gt_overlay")
preview_video = ToVideoNode(
    output_video_path=str(preview_video_path), frame_rate=5.0, name="to_video"
)

preview_pipeline.connect(
    (preview_data.outputs.cube, preview_selector.cube),
    (preview_data.outputs.wavelengths, preview_selector.wavelengths),
    (preview_selector.rgb_image, preview_overlay.rgb_image),
    (preview_data.outputs.mask, preview_overlay.mask),
    (preview_overlay.rgb_with_overlay, preview_video.rgb_image),
)

preview_pipeline

In [ ]:
video_datamodule = MultiNpzDataModule(
    splits=DataSplitConfig(splits_path=str(SPLITS_JSON.resolve())),
    universe_csv=str(UNIVERSE_CSV),
    batch_size=1,
)

preview_pipeline.to(device)
Predictor(pipeline=preview_pipeline, datamodule=video_datamodule).predict(collect_outputs=False)

if not preview_video_path.exists():
    raise RuntimeError(f"Preview video was not created: {preview_video_path}")
display(Video(str(preview_video_path), embed=IN_COLAB, width=640))

## 5 · Build the training pipeline

Nine nodes:

- `AnomalyDataNode` turns each batch into a float32 cube plus a wavelength vector, and maps
  the multi-class GT mask to a binary anomaly mask (classes in `normal_class_ids` become 0,
  everything else 1).
- `MinMaxNormalizer` scales the cube per channel; its running bounds are seeded by the
  statistical training phase.
- `CIRSelector` maps the bands nearest NIR 860 / Red 670 / Green 560 nm to R/G/B: a
  3-channel false-color image.
- `DinomalyDetector` wraps the frozen DINOv2 encoder plus the trainable bottleneck and
  decoder; it emits the pixel `scores` map, the per-frame `anomaly_score`, and a
  `training_loss`.
- `DinomalyTrainLossBridge` exposes that loss to the trainer.
- `QuantileBinaryDecider` binarizes the score map at the 99.5th percentile.
- `AnomalyDetectionMetrics` (IoU/Dice on the binarized map) drives checkpointing, and
  `AnomalyAUROCMetrics` streams pixel and image AUROC each epoch; both read the GT mask.
- `TensorBoardMonitorNode` logs the metrics.

The pipeline is saved to yaml right away: the trainrun in the next section references it by
path.

In [ ]:
pipeline = CuvisPipeline("dinomaly_lentils_cir")

data_node = AnomalyDataNode(normal_class_ids=[0], name="anomaly_data")
normalizer = MinMaxNormalizer(eps=1e-6, use_running_stats=True, max_initialization_frames=20)
selector = CIRSelector(
    nir_nm=860.0,
    red_nm=670.0,
    green_nm=560.0,
    norm_mode="running",
    running_warmup_frames=0,
    freeze_running_bounds_after_frames=20,
    name="cir_selector",
)
dinomaly = DinomalyDetector(
    encoder_name="dinov2reg_vit_base_14",
    bottleneck_dropout=0.2,
    decoder_depth=8,
    image_size=IMAGE_SIZE,
    crop_size=IMAGE_SIZE,
    use_center_crop=False,
    input_channels=3,
    name="dinomaly_detector",
)
loss_bridge = DinomalyTrainLossBridge(weight=1.0, name="dinomaly_train_loss")
decider = QuantileBinaryDecider(quantile=0.995, name="decider")
metrics_node = AnomalyDetectionMetrics(name="metrics_anomaly")
auroc_node = AnomalyAUROCMetrics(name="metrics_auroc")
tb = TensorBoardMonitorNode(output_dir=str(output_dir / "tensorboard"), run_name=pipeline.name)

pipeline.connect(
    (data_node.outputs.cube, normalizer.data),
    (normalizer.normalized, selector.cube),
    (data_node.outputs.wavelengths, selector.wavelengths),
    (selector.rgb_image, dinomaly.rgb_image),
    (dinomaly.outputs.training_loss, loss_bridge.raw_loss),
    (dinomaly.outputs.scores, decider.logits),
    (dinomaly.outputs.scores, metrics_node.logits),
    (decider.decisions, metrics_node.decisions),
    (data_node.outputs.mask, metrics_node.targets),
    (metrics_node.metrics, tb.metrics),
    (dinomaly.outputs.scores, auroc_node.scores),
    (data_node.outputs.mask, auroc_node.targets),
    (dinomaly.outputs.anomaly_score, auroc_node.anomaly_score),
)

pipeline.save_to_file(
    str(pipeline_yaml_path),
    metadata=PipelineMetadata(
        name=pipeline.name,
        description="Dinomaly on lentils VNIR NPZ, CIR selector (NIR 860 / Red 670 / Green 560 nm).",
        tags=["dinomaly", "anomalib", "lentils", "cir", "hyperspectral"],
        author="cuvis.ai",
    ),
)
print("Pipeline saved:", pipeline_yaml_path)

In [ ]:
pipeline

## 6 · Train, via a trainrun
Cuvis.AI uses PyTorch Lightning under the hood, and training is exposed through Cuvis.AI’s `TrainRun` interface rather than through manually assembled Lightning trainer objects.

A `TrainRunConfig` bundles everything required to reproduce a training run: the pipeline, referenced through the YAML file saved above; the data module; the training schedule; and the nodes that provide the loss functions and evaluation metrics.

`restore_trainrun` then executes the complete training workflow. This includes statistical initialization, such as determining the bounds of the `MinMax` normalizer; gradient-based training, during which the bottleneck and decoder are optimized while the DINOv2 encoder remains frozen; serialization of the trained pipeline; and validation and test passes that record the resulting metrics in TensorBoard.

The saved YAML therefore acts as a reproducible run specification and can be used to launch the same training workflow from the terminal:

```bash
uv run restore-trainrun --trainrun-path outputs/lentils_cir_run/trainrun.yaml --mode train
```

In [ ]:
trainrun = TrainRunConfig(
    name="dinomaly_lentils_cir",
    pipeline=pipeline_yaml_path.name,  # resolved relative to the trainrun yaml
    data=DataConfig(
        data_module="npz_multi",
        batch_size=1,
        num_workers=0,
        # splits_path points at the splits.json; core loads it (absolute, so it resolves
        # regardless of where the trainrun runs from). universe_csv is the universe lookup.
        splits=DataSplitConfig(splits_path=str(SPLITS_JSON.resolve())),
        params={"universe_csv": str(UNIVERSE_CSV.resolve())},
    ),
    training=TrainingConfig(
        seed=42,
        max_epochs=MAX_EPOCHS,
        accelerator="auto",
        devices=1,
        default_root_dir=str(output_dir),
        precision="32-true",
        enable_progress_bar=True,
        enable_checkpointing=True,
        log_every_n_steps=10,
        check_val_every_n_epoch=1,
        gradient_clip_val=0.1,
        optimizer=OptimizerConfig(name="adamw", lr=2e-3, weight_decay=1e-4, betas=[0.9, 0.999]),
        callbacks=CallbacksConfig(
            checkpoint=ModelCheckpointConfig(
                dirpath=str(output_dir / "checkpoints"),
                monitor="metrics_anomaly/iou",
                mode="max",
                save_top_k=1,
                save_last=True,
                filename="{epoch:02d}",
            )
        ),
    ),
    loss_nodes=["dinomaly_train_loss"],
    metric_nodes=["metrics_anomaly", "metrics_auroc"],
    unfreeze_nodes=["dinomaly_detector"],
    output_dir=str(output_dir),
)
trainrun.save_to_file(trainrun_yaml_path)
print("Trainrun saved:", trainrun_yaml_path)

restore_trainrun(trainrun_yaml_path, mode="train")

## 7 · What the run produced

`restore_trainrun` saved the trained pipeline itself (yaml plus weights under
`trained_models/`); there is nothing to persist by hand.

In [ ]:
trained_dir = output_dir / "trained_models"
for artifact in sorted(trained_dir.glob("*")):
    print(f"{artifact}  ({artifact.stat().st_size / 1e6:.0f} MB)")

trained_yaml = trained_dir / f"{pipeline.name}_restored.yaml"
assert trained_yaml.is_file(), f"expected trained pipeline at {trained_yaml}"

### Per-node timing

Load the saved pipeline back and profile one short inference pass over the test split. The
built-in profiler wraps every `node.forward()`, so the table shows where the per-frame budget
goes: the DINOv2 detector dominates, the selector and decider are cheap. Per-node latency is
independent of the trained weights (same graph, same shapes), so the 1-epoch smoke model times
the same as a fully trained one. On GPU we pass `synchronize_cuda=True` for accurate CUDA
wall-clock numbers.

In [ ]:
# Load the saved pipeline back and profile one short inference pass over the test split. The
# built-in profiler wraps each node.forward(), so the table shows where the per-frame budget goes.
from cuvis_ai_core.utils.node_registry import NodeRegistry

PROFILE_FRAMES = 12  # test frames to time; the first couple are discarded as warm-up

registry = NodeRegistry()
registry.register_plugin(str(Path("../../configs/plugins/dinomaly.yaml").resolve()))
trained_pipeline = CuvisPipeline.load_pipeline(
    str(trained_yaml),
    weights_path=str(trained_yaml.with_suffix(".pt")),
    device=str(device),
    node_registry=registry,
)
trained_pipeline.torch_layers.eval()

profile_datamodule = MultiNpzDataModule(
    splits=DataSplitConfig(splits_path=str(SPLITS_JSON.resolve())),
    universe_csv=str(UNIVERSE_CSV),
    batch_size=1,
)

trained_pipeline.set_profiling(
    enabled=True, synchronize_cuda=device.type == "cuda", reset=True, skip_first_n=2
)
Predictor(pipeline=trained_pipeline, datamodule=profile_datamodule).predict(
    max_batches=PROFILE_FRAMES, collect_outputs=False
)
print(trained_pipeline.format_profiling_summary(total_frames=PROFILE_FRAMES))

## 8 · Next

Evaluate the trained pipeline on the 180-frame test split with
**`lentils_inference_tutorial.ipynb`**, pointing its pipeline directory at
`outputs/lentils_cir_run/trained_models`. A 1-epoch smoke model will score modestly; for a
real model set `MAX_EPOCHS = 20` (or 50) in section 3 and rerun from section 6, either here or
with the `restore-trainrun` command above. TensorBoard logs live under
`outputs/lentils_cir_run/tensorboard`.